# FSOT 2.1 — Prediction Monitor (Kaggle mirror)

Public, zero-free-parameter seed engine (**pin D1D38A**) + preregistered prediction locks.

This notebook mirrors the monorepo prediction spine:
1. Load `fsot_compute.py` and validate seed constants
2. Load prereg + freeze + monitor registry
3. Report near-future survey watches
4. Optional online GWOSC probe

**Authority:** GitHub monorepo `FSOT-2.1-Lean` — this is a portable mirror, not a retune surface.


In [ ]:
from pathlib import Path
import json, hashlib, sys
import yaml

# Kaggle input path when dataset is attached; local fallback for offline smoke
CANDIDATES = [
    Path('/kaggle/input/fsot-prediction-monitor'),
    Path('/kaggle/input/fsot-prediction-monitor/dataset'),
    Path('.'),
    Path('dataset'),
]
ROOT = next((p for p in CANDIDATES if (p / 'fsot_compute.py').is_file() or (p / 'vendor' / 'fsot_compute.py').is_file()), Path('.'))
if (ROOT / 'vendor' / 'fsot_compute.py').is_file():
    ENGINE = ROOT / 'vendor' / 'fsot_compute.py'
    DATA = ROOT / 'data' if (ROOT / 'data').is_dir() else ROOT
else:
    ENGINE = ROOT / 'fsot_compute.py'
    DATA = ROOT
print('ROOT', ROOT)
print('ENGINE', ENGINE, 'sha256', hashlib.sha256(ENGINE.read_bytes()).hexdigest()[:12].upper())


In [ ]:
sys.path.insert(0, str(ENGINE.parent))
import fsot_compute as fc

sha = hashlib.sha256(ENGINE.read_bytes()).hexdigest().upper()
pin_ok = sha.startswith('D1D38A')
print('pin_prefix', sha[:6], 'pin_match', pin_ok)
print('S_cosm', float(fc.S_COSM))
print('S_quant', float(fc.S_QUANT))
print('K', float(fc.K))
print('N_eff sample (wave1 if available)')
try:
    rows = list(fc.wave1())
    for r in rows[:8]:
        print(' ', getattr(r, 'name', r), getattr(r, 'value', None), getattr(r, 'target', None))
except Exception as e:
    print('wave1 skip', e)
assert pin_ok, 'Engine pin must start with D1D38A — refuse silent retune'


In [ ]:
def load_yaml(name):
    p = DATA / name
    if not p.is_file():
        p = DATA / 'data' / name if (DATA / 'data').is_dir() else p
    return yaml.safe_load(p.read_text(encoding='utf-8')) if p.is_file() else {}

def load_json(name):
    for p in [DATA / name, DATA / 'data' / name]:
        if p.is_file():
            return json.loads(p.read_text(encoding='utf-8'))
    return {}

prereg = load_yaml('preregistered_predictions_manifest.yaml')
registry = load_yaml('prediction_monitor_registry.yaml')
freeze = load_json('toe_prereg_freeze.json')
monitor = load_json('prediction_monitor_report.json')
margin = load_json('margin_slim.json')

preds = prereg.get('predictions') or []
print(f"PRED count: {len(preds)}")
print(f"domains: {len({p.get('domain') for p in preds})}")
print(f"future_survey tagged: {sum(1 for p in preds if p.get('future_survey'))}")
print(f"T5 freeze: {freeze.get('freeze_id')} sha={str(freeze.get('bundle_sha256'))[:16]}…")
print(f"monitor watches: {(monitor.get('summary') or {}).get('watch_count')}")
print(f"green (slim): {margin.get('green_gate_pass_count')}/{margin.get('benchmark_file_count')}")


In [ ]:
import pandas as pd

rows = []
for w in (registry.get('watches') or []):
    dd = w.get('data_drop') or {}
    rows.append({
        'id': w.get('id'),
        'title': w.get('title'),
        'sector': w.get('sector'),
        'urgency': w.get('urgency'),
        'fsot_lock': w.get('fsot_lock'),
        'unit': w.get('unit'),
        'window': dd.get('window'),
        'facility': dd.get('facility'),
        'pred_ids': ','.join(w.get('pred_ids') or []),
    })
df = pd.DataFrame(rows)
display(df)

print('\nHigh-urgency near-term:')
display(df[df.urgency == 'high'][['id', 'title', 'window', 'fsot_lock']])


In [ ]:
# Optional: live GWOSC catalog size (requires internet on Kaggle)
import urllib.request
ONLINE = True  # set False for offline-only
if ONLINE:
    try:
        url = 'https://www.gwosc.org/eventapi/json/GWTC/'
        with urllib.request.urlopen(url, timeout=20) as resp:
            data = json.loads(resp.read().decode())
        events = data.get('events') or {}
        n = len(events) if isinstance(events, dict) else data.get('numRows')
        print('GWOSC GWTC event entries:', n)
        print('FSOT PRED-048: compact-binary panel residual ceiling 0.5% (see monorepo green gate)')
    except Exception as e:
        print('online probe skipped:', e)
else:
    print('online disabled')


## Kill criteria (do not retune)

- If a survey posterior **excludes** a frozen lock (e.g. wa, N_eff, H0 bridge), record a kill in the monorepo falsification registry.
- Never change `fsot_predicted` without a new `freeze_id`.
- Full multiprover + 472-domain atlas: clone GitHub `dappalumbo91/FSOT-2.1-Lean`.
